In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/Sanjoli2707/AI-Exam-Anxiety-Detector.git
%cd AI-Exam-Anxiety-Detector
!ls

In [ ]:
!grep -v pywin32 requirements.txt | xargs pip install

In [ ]:
!ls -lh requirements.txt

In [ ]:
!pip install torch transformers pandas scikit-learn numpy tqdm fastapi uvicorn

In [ ]:
!ls data

In [ ]:
import pandas as pd

df = pd.read_csv("data/mental_health_unbalanced.csv")

df.head()

In [ ]:
df['status'].value_counts()

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['status'])

df[['status','label']].head()

In [ ]:
label_encoder.classes_

In [ ]:
texts = df['text'].astype(str).tolist()
labels = df['label'].tolist()

In [ ]:
print(texts[:5])
print(labels[:5])

In [ ]:
!pip install transformers

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
df[['status','label']].head()

In [ ]:
label_encoder.classes_

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
encodings.keys()

In [ ]:
print(list(encodings.keys()))

In [ ]:
print(encodings['input_ids'][0])

In [ ]:
len(texts)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.1,
    random_state=42
)

In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
import torch

class MentalHealthDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(train_encodings, train_labels)
val_dataset = MentalHealthDataset(val_encodings, val_labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
model.save_pretrained("model/bert_anxiety_model")
tokenizer.save_pretrained("model/bert_anxiety_model")

In [ ]:
trainer.evaluate()

 Model Evaluation & Performance Analysis

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

model_path = "model/bert_anxiety_model"

model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

model.eval()   # switch to evaluation mode

In [ ]:
torch.no_grad()

In [ ]:
labels = ['Anxiety','Depression','Normal','Suicidal']

def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits).item()

    return labels[pred]

In [ ]:
print(predict("I feel calm and prepared for exams"))
print(predict("I cannot sleep because of exam stress"))
print(predict("I feel hopeless and exhausted"))
print(predict("I am extremely nervous about tomorrow's test"))

In [ ]:
tests = [
"I am worried about my exam tomorrow",
"I am extremely nervous about tomorrow's test",
"I cannot focus because of exam stress",
"I feel calm and prepared for my exam",
"I feel hopeless and tired of everything",
"I want to give up on everything",
"I studied well and feel confident",
"I am panicking about failing the exam"
]

for t in tests:
    print(t, "->", predict(t))